In [7]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

contrast_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)


class ClfDS(Dataset):
    def __init__(self, folder):
        self.folder = folder
        classes=[]
        images=[]
        for cls in os.listdir(self.folder):
          for files in os.listdir(os.path.join(self.folder, cls)):
            img_path =os.path.join(self.folder, cls, files)
            images.append(img_path)
            classes.append(cls)
        self.id2label = {idx: label for idx, label in enumerate(sorted(set(classes)))}
        self.label2id = {label: idx for idx, label in enumerate(sorted(set(classes)))}
        self.classes = classes
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        id=self.classes[idx]
        img = Image.open(img_path)
        img = contrast_transforms(img)

        return img, self.label2id[id]

In [8]:
from torch.utils.data import DataLoader
import os
train_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\train'), batch_size=32, shuffle=True)
val_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\valid'), batch_size=32, shuffle=False)

In [9]:
next(iter(train_dl))

[tensor([[[[0.9137, 0.9137, 0.9176,  ..., 0.9961, 0.9961, 0.9961],
           [0.9373, 0.9373, 0.9412,  ..., 0.9922, 0.9922, 0.9922],
           [0.9725, 0.9725, 0.9765,  ..., 0.9922, 0.9882, 0.9882],
           ...,
           [0.5373, 0.5412, 0.5451,  ..., 0.9490, 0.9490, 0.9490],
           [0.6745, 0.6745, 0.6745,  ..., 0.8824, 0.8863, 0.8902],
           [0.6745, 0.6706, 0.6706,  ..., 0.8745, 0.8784, 0.8824]],
 
          [[0.1804, 0.1804, 0.1843,  ..., 0.3176, 0.3176, 0.3176],
           [0.2118, 0.2118, 0.2157,  ..., 0.3176, 0.3216, 0.3216],
           [0.2549, 0.2549, 0.2549,  ..., 0.3216, 0.3255, 0.3255],
           ...,
           [0.1922, 0.1922, 0.1882,  ..., 0.2627, 0.2588, 0.2588],
           [0.2941, 0.2941, 0.2902,  ..., 0.2392, 0.2392, 0.2392],
           [0.3216, 0.3176, 0.3137,  ..., 0.2235, 0.2235, 0.2196]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0118, 0.0118, 0.0078],
           [0.0039, 0.0039, 0.0039,  ..., 0.0078, 0.0078, 0.0000],
           [0.0118, 0.01

In [10]:
next(iter(val_dl))

[tensor([[[[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [0.9451, 0.9529, 0.9647,  ..., 1.0000, 0.9961, 1.0000],
           [0.8980, 0.9020, 0.9098,  ..., 1.0000, 0.9961, 0.9961],
           ...,
           [0.9804, 0.9765, 0.9804,  ..., 1.0000, 1.0000, 1.0000],
           [0.8667, 0.8627, 0.8745,  ..., 1.0000, 1.0000, 1.0000],
           [0.6784, 0.6706, 0.6706,  ..., 1.0000, 1.0000, 1.0000]],
 
          [[0.6196, 0.6157, 0.6000,  ..., 0.3529, 0.3490, 0.3569],
           [0.4118, 0.4196, 0.4275,  ..., 0.3529, 0.3490, 0.3529],
           [0.3333, 0.3373, 0.3412,  ..., 0.3569, 0.3490, 0.3529],
           ...,
           [0.9569, 0.9490, 0.9373,  ..., 0.3137, 0.3137, 0.3137],
           [0.7804, 0.7765, 0.7843,  ..., 0.3137, 0.3137, 0.3137],
           [0.5686, 0.5608, 0.5647,  ..., 0.3137, 0.3137, 0.3137]],
 
          [[0.2941, 0.2902, 0.2784,  ..., 0.0078, 0.0078, 0.0196],
           [0.0902, 0.0980, 0.1059,  ..., 0.0078, 0.0000, 0.0118],
           [0.0157, 0.01

In [12]:
import math, json, os, torch, torch.nn as nn, torch.nn.functional as F
import pytorch_lightning as pl, timm
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

torch.set_float32_matmul_precision("high")
EPOCHS = 10          # keyin 100
N_CLS = len(set(train_dl.dataset.classes))


class CACLoss(nn.Module):
    def __init__(self, n_cls, alpha=10.0, lam=0.1):
        super().__init__()
        self.n_cls, self.lam = n_cls, lam
        self.register_buffer("anchors", torch.eye(n_cls) * alpha)

    def dist(self, emb):
        return torch.cdist(emb.float(), self.anchors)

    def forward(self, emb, y):
        d = self.dist(emb)
        onehot = F.one_hot(y, self.n_cls).bool()
        d_true = d.gather(1, y[:, None])
        anchor = d_true.mean()
        tuplet = torch.log1p(torch.exp(d_true - d).masked_fill(onehot, 0).sum(1)).mean()
        return tuplet + self.lam * anchor


class LitCAC(pl.LightningModule):
    def __init__(self, n_cls, lr=3e-4):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0)
        self.proj = nn.Linear(192, n_cls)        # CAC: embedding o'lchami = class soni
        self.crit = CACLoss(n_cls)
        self._p, self._t = [], []

    def forward(self, x):
        return self.proj(self.backbone(x))

    def training_step(self, batch, _):
        x, y = batch
        loss = self.crit(self(x), y)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        emb = self(x)
        self.log("val_loss", self.crit(emb, y), prog_bar=True)
        self._p.append(self.crit.dist(emb).argmin(1).cpu())     # eng yaqin anchor
        self._t.append(y.cpu())

    def on_validation_epoch_end(self):
        p, t = torch.cat(self._p).numpy(), torch.cat(self._t).numpy()
        self._p.clear(); self._t.clear()
        pm, rm, fm, _ = precision_recall_fscore_support(t, p, average="macro", zero_division=0)
        pw, rw, fw, _ = precision_recall_fscore_support(t, p, average="weighted", zero_division=0)
        self.log_dict({"val_acc": accuracy_score(t, p),
                       "val_prec_macro": pm, "val_rec_macro": rm, "val_f1_macro": fm,
                       "val_prec_w": pw, "val_rec_w": rw, "val_f1_w": fw})

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        return [opt], [torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs)]

pl.seed_everything(42)
model = LitCAC(N_CLS)
ckpt = ModelCheckpoint(monitor="val_f1_macro", mode="max")
trainer = pl.Trainer(max_epochs=EPOCHS, accelerator="gpu", precision="16-mixed",
                     logger=CSVLogger("logs", name=f"cac_{EPOCHS}ep"),
                     callbacks=[ckpt], log_every_n_steps=1)
trainer.fit(model, train_dl, val_dl)

os.makedirs("results", exist_ok=True)
res = {k: float(v) for k, v in trainer.callback_metrics.items() if k.startswith("val_")}
res["best_f1_macro"] = float(ckpt.best_model_score)
json.dump(res, open(f"results/cac_{EPOCHS}ep.json", "w"), indent=2)
print(res)

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone │ VisionTransformer │  5.5 M │ train │     0 │
│ 1 │ proj     │ Linear            │ 33.2 K │ train │     0 │
│ 2 │ crit     │ CACLoss           │      0 │ train │     0 │
└───┴──────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 5.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.6 M                                                                                                
Total estimated model params size (MB): 22.230                                                                     
Modules in train mode: 278                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.p
y:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.p
y:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value 
of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=10` reached.


{'val_loss': 1.2886062860488892, 'val_acc': 0.8378499746322632, 'val_prec_macro': 0.7704562544822693, 'val_rec_macro': 0.7593100070953369, 'val_f1_macro': 0.757172167301178, 'val_prec_w': 0.8381003737449646, 'val_rec_w': 0.8378499746322632, 'val_f1_w': 0.8352435231208801, 'best_f1_macro': 0.757172167301178}


In [15]:
torch.save(model, "CACloss_VIT_clf.pth")

In [16]:
model=torch.load(r"C:\Users\Administrator\Desktop\Coding\CACloss_VIT_clf.pth", weights_only=False)
model.eval()

LitCAC(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=192, out_features=768, bias=True)
          (act): 

In [17]:
clf_model = model.cuda()
import torch
import torch.nn.functional as F

from sklearn.metrics import classification_report
import numpy as np

all_preds = []
all_labels = []

for batch in val_dl:
    images, labels = batch
    val_out = clf_model(images.cuda())

    probs = F.softmax(val_out, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()
    labels = labels.cpu().numpy()

    preds = [val_dl.dataset.id2label[idx] for idx in preds]
    labels = [val_dl.dataset.id2label[idx] for idx in labels]

    all_preds.extend(preds)
    all_labels.extend(labels)

print(classification_report(all_labels, all_preds))

                          precision    recall  f1-score   support

                n_1_1050       0.78      0.82      0.80        17
              n_1_1050-2       0.50      0.52      0.51        27
              n_1_1050-3       0.88      0.90      0.89       139
                 n_1_300       0.91      0.85      0.88        79
               n_1_300-2       0.79      0.83      0.81        71
               n_1_300-3       0.93      0.91      0.92       182
                 n_1_600       0.90      0.95      0.93        40
               n_1_600-2       0.89      0.81      0.85        48
               n_1_600-3       0.91      0.91      0.91       163
                n_2_1050       0.89      0.89      0.89         9
              n_2_1050-2       0.70      0.66      0.68        32
              n_2_1050-3       0.91      0.91      0.91       106
                 n_2_300       0.87      0.90      0.88        81
               n_2_300-2       0.76      0.74      0.75        65
         

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital